runs once

In [1]:
%run dataframes.ipynb

C:\Users\SafinSharma\AppData\Local\Temp\ipykernel_18768\4049516586.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
C:\Users\SafinSharma\AppData\Local\Temp\ipykernel_18768\4049516586.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


In [2]:
def standardize_dataset(df, dataset_name):

    name = dataset_name.lower()

    # =====================================================
    # METR-LA TRAFFIC
    # =====================================================
    if "traffic" in name or "metr" in name:

        timestamp_col = df.columns[0]  # Unnamed: 0

        output = df.melt(
            id_vars=timestamp_col,
            var_name="unique_id",
            value_name="y"
        )

        output = output.rename(
            columns={
                timestamp_col: "ds"
            }
        )

        output["ds"] = pd.to_datetime(output["ds"])

        output["unique_id"] = output["unique_id"].astype(str)

        return output[
            ["unique_id", "ds", "y"]
        ]

    # =====================================================
    # ELECTRICITY
    # =====================================================
    elif "energy" in name or "electricity" in name:

        timestamp_col = df.columns[0]  # Unnamed: 0

        output = df.melt(
            id_vars=timestamp_col,
            var_name="unique_id",
            value_name="y"
        )

        output = output.rename(
            columns={
                timestamp_col: "ds"
            }
        )

        output["ds"] = pd.to_datetime(output["ds"])

        return output[
            ["unique_id", "ds", "y"]
        ]

    # =====================================================
    # FAVORITA TRAIN
    # =====================================================
    elif "favorita" in name and "train" in name:

        output = df.copy()

        output["unique_id"] = (
            output["store_nbr"].astype(str)
            + "_"
            + output["item_nbr"].astype(str)
        )

        output["ds"] = pd.to_datetime(
            output["date"]
        )

        output["y"] = output["unit_sales"]

        keep_cols = [
            "unique_id",
            "ds",
            "y"
        ]

        if "onpromotion" in output.columns:
            keep_cols.append("onpromotion")

        return output[keep_cols]

    # =====================================================
    # FAVORITA TEST
    # =====================================================
    elif "favorita" in name and "test" in name:

        output = df.copy()

        output["unique_id"] = (
            output["store_nbr"].astype(str)
            + "_"
            + output["item_nbr"].astype(str)
        )

        output["ds"] = pd.to_datetime(
            output["date"]
        )

        return output[
            [
                "unique_id",
                "ds",
                "onpromotion"
            ]
        ]

    # =====================================================
    # FAVORITA TRANSACTIONS
    # =====================================================
    elif "favorita" in name and "transactions" in name:

        output = df.copy()

        output["unique_id"] = (
            "store_"
            + output["store_nbr"].astype(str)
        )

        output["ds"] = pd.to_datetime(
            output["date"]
        )

        output["y"] = output["transactions"]

        return output[
            ["unique_id", "ds", "y"]
        ]

    # =====================================================
    # M5
    # =====================================================
    elif "m5" in name or "df" in name:

        day_cols = [
            c
            for c in df.columns
            if c.startswith("d_")
        ]

        output = df.melt(
            id_vars=[
                "id",
                "item_id",
                "dept_id",
                "cat_id",
                "store_id",
                "state_id"
            ],
            value_vars=day_cols,
            var_name="ds",
            value_name="y"
        )

        output = output.rename(
            columns={
                "id": "unique_id"
            }
        )

        return output[
            [
                "unique_id",
                "ds",
                "y",
                "item_id",
                "dept_id",
                "cat_id",
                "store_id",
                "state_id"
            ]
        ]

    # =====================================================
    # M4
    # =====================================================
    elif "m4" in name:

        id_col = df.columns[0]  # V1

        value_cols = [
            c
            for c in df.columns
            if c != id_col
        ]

        output = df.melt(
            id_vars=id_col,
            value_vars=value_cols,
            var_name="ds",
            value_name="y"
        )

        output = output.rename(
            columns={
                id_col: "unique_id"
            }
        )

        output = output.dropna(
            subset=["y"]
        )

        output["ds"] = (
            output["ds"]
            .str.replace("V", "", regex=False)
            .astype(int)
        )

        return output[
            ["unique_id", "ds", "y"]
        ]

    # =====================================================
    # FAOSTAT AGRI
    # =====================================================
    elif "agri" in name or "faostat" in name:

        output = df.copy()

        output["unique_id"] = (
            output["Area"].astype(str)
            + "_"
            + output["Item"].astype(str)
            + "_"
            + output["Element"].astype(str)
        )

        output["ds"] = output["Year"]

        output["y"] = output["Value"]

        output = output.dropna(
            subset=["y"]
        )

        return output[
            [
                "unique_id",
                "ds",
                "y",
                "Area",
                "Item",
                "Element",
                "Unit"
            ]
        ]

    else:

        raise ValueError(
            f"Dataset type not recognized: {dataset_name}"
        )

In [3]:
standardized = {}

for name, df in datasets.items():

    try:

        standardized[name] = standardize_dataset(
            df,
            name
        )

    except Exception as e:

        print(
            f"{name} failed:"
        )

        print(e)

In [4]:
def standardize_traffic(df):
    df = df.rename(columns={"Unnamed: 0": "ds"})

    return df.melt(
        id_vars=["ds"],
        var_name="unique_id",
        value_name="y"
    )

In [5]:
def standardize_energy(df):
    df = df.rename(columns={"Unnamed: 0": "ds"})

    return df.melt(
        id_vars=["ds"],
        var_name="unique_id",
        value_name="y"
    )

In [6]:
def standardize_favorita(train_df, transactions_df):
    train_df["ds"] = pd.to_datetime(train_df["date"])

    train_df["unique_id"] = (
        train_df["store_nbr"].astype(str)
        + "_"
        + train_df["item_nbr"].astype(str)
    )

    transactions_df["ds"] = pd.to_datetime(transactions_df["date"])

    df = train_df.merge(
        transactions_df[["ds","store_nbr","transactions"]],
        on=["ds","store_nbr"],
        how="left"
    )

    return df.rename(columns={"unit_sales":"y"})[
        [
            "unique_id",
            "ds",
            "y",
            "onpromotion",
            "transactions"
        ]
    ]

In [7]:
def standardize_agri(df):

    df["unique_id"] = (
        df["Item"].astype(str)
        + "_"
        + df["Element"].astype(str)
    )

    return pd.DataFrame({
        "unique_id": df["unique_id"],
        "ds": pd.to_datetime(df["Year"], format="%Y"),
        "y": df["Value"]
    })

In [8]:
def standardize_m4(df, prefix):
    
    series_col = df.columns[0]

    long_df = df.melt(
        id_vars=[series_col],
        var_name="t",
        value_name="y"
    )

    long_df = long_df.dropna(subset=["y"])

    long_df = long_df.rename(
        columns={series_col: "unique_id"}
    )

    long_df["ds"] = (
        long_df.groupby("unique_id")
               .cumcount()
               .add(1)
    )

    return long_df[
        ["unique_id","ds","y"]
    ]

In [9]:
def standardize_m5(df):

    id_cols = [
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id"
    ]

    value_cols = [
        c for c in df.columns
        if c.startswith("d_")
    ]

    long_df = df.melt(
        id_vars=id_cols,
        value_vars=value_cols,
        var_name="ds",
        value_name="y"
    )

    long_df = long_df.rename(
        columns={"id":"unique_id"}
    )

    return long_df[
        ["unique_id","ds","y"]
    ]

In [10]:
# ============================================================
# TRAFFIC
# ============================================================

def standardize_traffic(df):
    df = df.copy()
    df = df.rename(columns={"Unnamed: 0": "ds"})

    return (
        df.melt(
            id_vars=["ds"],
            var_name="unique_id",
            value_name="y"
        )
        .assign(ds=lambda x: pd.to_datetime(x["ds"]))
        .sort_values(["unique_id", "ds"])
        .reset_index(drop=True)
    )


# ============================================================
# ENERGY
# ============================================================

def standardize_energy(df):
    df = df.copy()
    df = df.rename(columns={"Unnamed: 0": "ds"})

    return (
        df.melt(
            id_vars=["ds"],
            var_name="unique_id",
            value_name="y"
        )
        .assign(ds=lambda x: pd.to_datetime(x["ds"]))
        .sort_values(["unique_id", "ds"])
        .reset_index(drop=True)
    )


# ============================================================
# FAVORITA TRAIN
# ============================================================

def standardize_favorita_train(train_df, transactions_df):

    df = train_df.copy()

    df["ds"] = pd.to_datetime(df["date"])

    df["unique_id"] = (
        df["store_nbr"].astype(str)
        + "_"
        + df["item_nbr"].astype(str)
    )

    trans = transactions_df.copy()
    trans["ds"] = pd.to_datetime(trans["date"])

    df = df.merge(
        trans[["ds", "store_nbr", "transactions"]],
        how="left",
        on=["ds", "store_nbr"]
    )

    return (
        df.rename(columns={"unit_sales": "y"})
        [[
            "unique_id",
            "ds",
            "y",
            "onpromotion",
            "transactions"
        ]]
        .sort_values(["unique_id", "ds"])
        .reset_index(drop=True)
    )


# ============================================================
# FAVORITA TEST
# ============================================================

def standardize_favorita_test(test_df):

    df = test_df.copy()

    df["ds"] = pd.to_datetime(df["date"])

    df["unique_id"] = (
        df["store_nbr"].astype(str)
        + "_"
        + df["item_nbr"].astype(str)
    )

    return (
        df[[
            "id",
            "unique_id",
            "ds",
            "onpromotion"
        ]]
        .sort_values(["unique_id", "ds"])
        .reset_index(drop=True)
    )


# ============================================================
# AGRICULTURE
# ============================================================

def standardize_agri(df):

    tmp = df.copy()

    tmp["unique_id"] = (
        tmp["Item"].astype(str)
        + "_"
        + tmp["Element"].astype(str)
    )

    return (
        pd.DataFrame({
            "unique_id": tmp["unique_id"],
            "ds": pd.to_datetime(tmp["Year"], format="%Y"),
            "y": pd.to_numeric(tmp["Value"], errors="coerce"),
            "unit": tmp["Unit"]
        })
        .sort_values(["unique_id", "ds"])
        .reset_index(drop=True)
    )


# ============================================================
# M4
# ============================================================

def standardize_m4(df):

    df = df.copy()

    uid_col = df.columns[0]

    long_df = df.melt(
        id_vars=[uid_col],
        var_name="step",
        value_name="y"
    )

    long_df = long_df.dropna(subset=["y"])

    long_df = long_df.rename(
        columns={uid_col: "unique_id"}
    )

    long_df["ds"] = (
        long_df.groupby("unique_id")
               .cumcount()
               .add(1)
    )

    return (
        long_df[[
            "unique_id",
            "ds",
            "y"
        ]]
        .reset_index(drop=True)
    )


# ============================================================
# M5
# ============================================================

def standardize_m5(df):

    df = df.copy()

    meta_cols = [
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id"
    ]

    day_cols = [
        c for c in df.columns
        if c.startswith("d_")
    ]

    long_df = df.melt(
        id_vars=meta_cols,
        value_vars=day_cols,
        var_name="ds",
        value_name="y"
    )

    long_df = long_df.rename(
        columns={"id": "unique_id"}
    )

    return (
        long_df.sort_values(["unique_id", "ds"])
        .reset_index(drop=True)
    )


# ============================================================
# STANDARDIZE EVERYTHING
# ============================================================

std_traffic_small = standardize_traffic(traffic_small)
std_traffic_medium = standardize_traffic(traffic_medium)
std_traffic_large = standardize_traffic(traffic_large)

std_energy_small = standardize_energy(energy_small)
std_energy_medium = standardize_energy(energy_medium)
std_energy_large = standardize_energy(energy_large)

std_favorita_small_train = standardize_favorita_train(
    favorita_small_train,
    favorita_small_transactions
)

std_favorita_medium_train = standardize_favorita_train(
    favorita_medium_train,
    favorita_medium_transactions
)

std_favorita_large_train = standardize_favorita_train(
    favorita_large_train,
    favorita_large_transactions
)

std_favorita_small_test = standardize_favorita_test(
    favorita_small_test
)

std_favorita_medium_test = standardize_favorita_test(
    favorita_medium_test
)

std_favorita_large_test = standardize_favorita_test(
    favorita_large_test
)

std_agri_small = standardize_agri(agri_small)
std_agri_medium = standardize_agri(agri_medium)
std_agri_large = standardize_agri(agri_large)

std_m4_daily_small = standardize_m4(m4_daily_small)
std_m4_hourly_small = standardize_m4(m4_hourly_small)
std_m4_monthly_small = standardize_m4(m4_monthly_small)
std_m4_quarterly_small = standardize_m4(m4_quarterly_small)
std_m4_weekly_small = standardize_m4(m4_weekly_small)
std_m4_yearly_small = standardize_m4(m4_yearly_small)

std_m4_daily_medium = standardize_m4(m4_daily_medium)
std_m4_hourly_medium = standardize_m4(m4_hourly_medium)
std_m4_monthly_medium = standardize_m4(m4_monthly_medium)
std_m4_quarterly_medium = standardize_m4(m4_quarterly_medium)
std_m4_weekly_medium = standardize_m4(m4_weekly_medium)
std_m4_yearly_medium = standardize_m4(m4_yearly_medium)

std_m4_daily_large = standardize_m4(m4_daily_large)
std_m4_hourly_large = standardize_m4(m4_hourly_large)
std_m4_monthly_large = standardize_m4(m4_monthly_large)
std_m4_quarterly_large = standardize_m4(m4_quarterly_large)
std_m4_weekly_large = standardize_m4(m4_weekly_large)
std_m4_yearly_large = standardize_m4(m4_yearly_large)

std_m5_train_eval_small = standardize_m5(
    m5_train_eval_small
)

std_m5_train_validate_small = standardize_m5(
    m5_train_validate_small
)

std_m5_train_eval_medium = standardize_m5(
    m5_train_eval_medium
)

std_m5_train_validate_medium = standardize_m5(
    m5_train_validate_medium
)

std_m5_train_eval_large = standardize_m5(
    m5_train_eval_large
)

std_m5_train_validate_large = standardize_m5(
    m5_train_validate_large
)


In [11]:
print(standardized)

{'traffic_small':        unique_id                  ds          y
0         773869 2012-03-01 00:00:00  64.375000
1         773869 2012-03-01 00:05:00  62.666667
2         773869 2012-03-01 00:10:00  64.000000
3         773869 2012-03-01 00:15:00   0.000000
4         773869 2012-03-01 00:20:00   0.000000
...          ...                 ...        ...
709384    769373 2012-03-12 21:10:00  64.125000
709385    769373 2012-03-12 21:15:00  63.333333
709386    769373 2012-03-12 21:20:00  63.750000
709387    769373 2012-03-12 21:25:00  62.500000
709388    769373 2012-03-12 21:30:00  62.000000

[709389 rows x 3 columns], 'traffic_medium':         unique_id                  ds          y
0          773869 2012-03-01 00:00:00  64.375000
1          773869 2012-03-01 00:05:00  62.666667
2          773869 2012-03-01 00:10:00  64.000000
3          773869 2012-03-01 00:15:00   0.000000
4          773869 2012-03-01 00:20:00   0.000000
...           ...                 ...        ...
3547147    769373

In [12]:
standardized = {
    f"std_{name}": df
    for name, df in standardized.items()
}

In [13]:
print(standardized)

{'std_traffic_small':        unique_id                  ds          y
0         773869 2012-03-01 00:00:00  64.375000
1         773869 2012-03-01 00:05:00  62.666667
2         773869 2012-03-01 00:10:00  64.000000
3         773869 2012-03-01 00:15:00   0.000000
4         773869 2012-03-01 00:20:00   0.000000
...          ...                 ...        ...
709384    769373 2012-03-12 21:10:00  64.125000
709385    769373 2012-03-12 21:15:00  63.333333
709386    769373 2012-03-12 21:20:00  63.750000
709387    769373 2012-03-12 21:25:00  62.500000
709388    769373 2012-03-12 21:30:00  62.000000

[709389 rows x 3 columns], 'std_traffic_medium':         unique_id                  ds          y
0          773869 2012-03-01 00:00:00  64.375000
1          773869 2012-03-01 00:05:00  62.666667
2          773869 2012-03-01 00:10:00  64.000000
3          773869 2012-03-01 00:15:00   0.000000
4          773869 2012-03-01 00:20:00   0.000000
...           ...                 ...        ...
3547147  

In [14]:
from pathlib import Path

# create export folder
output_dir = Path("standardized_exports")
output_dir.mkdir(exist_ok=True)

# save all standardized datasets
for name, df in standardized.items():
    file_path = output_dir / f"{name}.csv"
    df.to_csv(file_path, index=False)
    print(f"Saved: {file_path}")

Saved: standardized_exports\std_traffic_small.csv
Saved: standardized_exports\std_traffic_medium.csv
Saved: standardized_exports\std_traffic_large.csv
Saved: standardized_exports\std_energy_small.csv
Saved: standardized_exports\std_energy_medium.csv
Saved: standardized_exports\std_energy_large.csv
Saved: standardized_exports\std_favorita_small_train.csv
Saved: standardized_exports\std_favorita_small_test.csv
Saved: standardized_exports\std_favorita_small_transactions.csv
Saved: standardized_exports\std_favorita_medium_train.csv
Saved: standardized_exports\std_favorita_medium_test.csv
Saved: standardized_exports\std_favorita_medium_transactions.csv
Saved: standardized_exports\std_favorita_large_train.csv
Saved: standardized_exports\std_favorita_large_test.csv
Saved: standardized_exports\std_favorita_large_transactions.csv
Saved: standardized_exports\std_agri_small.csv
Saved: standardized_exports\std_agri_medium.csv
Saved: standardized_exports\std_agri_large.csv
Saved: standardized_expor